In [0]:
-- Creates rules that check against a retrieved data frame before data writes to the lakehouse
-- Replace the schema when necessary.
CREATE TABLE IF NOT EXISTS analytics_lakehouse.gold.data_quality_checks (
  table_name STRING,
  rule_name STRING,
  rule_description STRING,
  sql_expression STRING,
  severity STRING,
  check_type STRING,
  is_active BOOLEAN,
  data_source STRING
);

-- check_type guide
--   WHERE  rd_df.where(sql_expression).count()  (counts rows matching the condition (failures))
--   UNIQUE rd_df.groupBy(sql_expression).count().filter(col('count') > 1).count()  (counts duplicate groups)
 
MERGE INTO analytics_lakehouse.gold.data_quality_checks AS target
USING (
  SELECT * FROM (
    VALUES
  --------------------
  -- VEHICLE LISTINGS
  --------------------
  -- daily_listings (SELECT * from silver, incremental merge — same columns as silver)
    ('daily_listings', 'pk_not_null', 'Primary key must not be null', 'transaction_listing_id IS NULL', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),
    ('daily_listings', 'pk_unique', 'Primary key must be unique', 'transaction_listing_id', 'CRITICAL', 'UNIQUE', true, 'denver_vehicle_listings'),
    ('daily_listings', 'listing_id_not_null', 'Listing ID must not be null', 'listing_id IS NULL', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),
    ('daily_listings', 'retrieval_date_not_null', 'Retrieval date must not be null', 'retrieval_date IS NULL', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),
    ('daily_listings', 'retrieval_date_not_future', 'Retrieval date must not be in the future', 'retrieval_date > current_date()', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),
    ('daily_listings', 'vin_not_null', 'VIN must not be null', 'vin IS NULL', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),
    ('daily_listings', 'year_valid_range', 'Year must be between 1990 and next year', 'year < 1990 OR year > YEAR(current_date()) + 1', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),
    ('daily_listings', 'make_not_null', 'Make must not be null', 'make IS NULL', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),
    ('daily_listings', 'list_price_positive', 'List price must be positive when present', 'list_price IS NOT NULL AND list_price <= 0', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),
    ('daily_listings', 'mileage_non_negative', 'Mileage must be non-negative when present', 'mileage IS NOT NULL AND mileage < 0', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),
    ('daily_listings', 'days_on_market_non_negative', 'Days on market must be non-negative', 'days_on_market IS NOT NULL AND days_on_market < 0', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),

    -- vehicles (one row per vehicle_id, latest retrieval_date wins)
    ('vehicles', 'pk_not_null', 'Primary key must not be null', 'vehicle_id IS NULL', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),
    ('vehicles', 'pk_unique', 'Primary key must be unique — one row per vehicle', 'vehicle_id', 'CRITICAL', 'UNIQUE', true, 'denver_vehicle_listings'),
    ('vehicles', 'vin_not_null', 'VIN must not be null', 'vin IS NULL', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),
    ('vehicles', 'vin_unique', 'VIN should be unique across vehicles', 'vin', 'WARNING', 'UNIQUE', true, 'denver_vehicle_listings'),
    ('vehicles', 'year_valid_range', 'Year must be between 1990 and next year', 'year < 1990 OR year > YEAR(current_date()) + 1', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),
    ('vehicles', 'make_not_null', 'Make must not be null', 'make IS NULL', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),
    ('vehicles', 'model_not_null', 'Model must not be null', 'model IS NULL', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),
    ('vehicles', 'list_price_positive', 'List price must be positive when present', 'list_price IS NOT NULL AND list_price <= 0', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),
    ('vehicles', 'mileage_non_negative', 'Mileage must be non-negative when present', 'mileage IS NOT NULL AND mileage < 0', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),
    ('vehicles', 'dealer_id_not_null', 'Dealer FK must not be null', 'dealer_id IS NULL', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),

    -- dealers (one row per dealer_id, latest retrieval_date wins)
    ('dealers', 'pk_not_null', 'Primary key must not be null', 'dealer_id IS NULL', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),
    ('dealers', 'pk_unique', 'Primary key must be unique — one row per dealer', 'dealer_id', 'CRITICAL', 'UNIQUE', true, 'denver_vehicle_listings'),
    ('dealers', 'dealer_name_not_null', 'Dealer name must not be null', 'dealer_name IS NULL', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),
    ('dealers', 'dealer_state_valid', 'Dealer state must be a 2-letter code', 'dealer_state IS NOT NULL AND LENGTH(dealer_state) != 2', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),
    ('dealers', 'dealer_latitude_valid', 'Dealer latitude must be in valid US range', 'dealer_latitude IS NOT NULL AND (dealer_latitude < 24 OR dealer_latitude > 50)', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),
    ('dealers', 'dealer_longitude_valid', 'Dealer longitude must be in valid US range', 'dealer_longitude IS NOT NULL AND (dealer_longitude < -125 OR dealer_longitude > -66)', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),
    ('dealers', 'dealer_rating_range', 'Dealer average rating must be between 0 and 5', 'dealer_average_rating IS NOT NULL AND (dealer_average_rating < 0 OR dealer_average_rating > 5)', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),
    ('dealers', 'dealer_review_count_non_negative', 'Review count must be non-negative', 'dealer_review_count IS NOT NULL AND dealer_review_count < 0', 'WARNING', 'WHERE', true, 'denver_vehicle_listings')
  ) AS v (table_name, rule_name, rule_description, sql_expression, severity, check_type, is_active, data_source)
) AS source
ON target.table_name = source.table_name
   AND target.rule_name = source.rule_name
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *